### Chat History with Memory 🧠


### Load ENV file

In [ ]:
from pprint import pprint
from dotenv import load_dotenv
load_dotenv('../.env')


### Instantiate LLMs

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
   base_url="http://localhost:11434",
   model="qwen2.5:latest",
   temperature=0.5,
   max_tokens=250
)

llm2 = ChatOllama(
   base_url="http://localhost:11434",
   model="llama3.2:latest",
   temperature=0.5,
   max_tokens=250
)

In [ ]:
%pip install langchain_community

## Message History with ChatMessageHistory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [ ]:


template = ChatPromptTemplate.from_messages([
    ("human", "{prompt}"),
    ("placeholder", "{history}"),
])
chain = template | llm2 | StrOutputParser()

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="prompt",
    history_messages_key="history"
)

session_id = "Karthik"
get_session_history(session_id).clear()
response1 = history.invoke({"prompt": "What is the benefit of running LLM in local machine"},
                          config={"configurable": {"session_id": session_id}})

response2 = history.invoke({"prompt": "How about for cloud"},
                          config={"configurable": {"session_id": session_id}})
pprint(response1)
print("\n\n")
pprint(response2)

### ChatMessage History with SqlChatMessageHistory

In [9]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

def get_session_history(session_id) -> SQLChatMessageHistory:
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_history.db"
    )

template = ChatPromptTemplate.from_messages([
    ("human", "{prompt}"),
    ("placeholder", "{history}"),
])
chain = template | llm | StrOutputParser()

store = {}

history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="prompt",
    history_messages_key="history"
)

session_id = "Karthik"
get_session_history(session_id).clear()
response1 = history.invoke({"prompt": "What is the distance between earth and sun?"},
                          config = {"configurable": {"session_id": session_id}})

response2 = history.invoke({"prompt": "How about the distance to the moon"},
                          config = {"configurable": {"session_id": session_id}})
pprint(response1)
print("\n\n")
pprint(response2)

('The average distance between Earth and the Sun is approximately 93 million '
 'miles (150 million kilometers). This average distance defines one '
 'Astronomical Unit (AU), which is a commonly used unit of measurement in '
 'astronomy for distances within our solar system. The actual distance can '
 "vary slightly due to Earth's elliptical orbit, ranging from about 91.4 "
 'million miles (147 million km) at perihelion to about 94.5 million miles '
 '(152 million km) at aphelion.')



''
